In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn as sl
import matplotlib.pyplot as plt

In [2]:
data = pd.read_csv('Rejestr_Produktow_Leczniczych_calosciowy_stan_na_dzien_20250425.csv', sep=";")

In [3]:
data.shape

(22305, 33)

In [4]:
#data.info

In [5]:
#data.info()

Sprawdzenie jak wygląda nasz zbiór danych, jest tak dużo kolumn, że podzieliłam je na 4 części, aby  móc lepiej się przyjrzeć

In [6]:
col = data.columns
col1 = col[0:8]
col2 = col[8:17]
col3 = col[17:25]
col4 = col[25:33]

Sprawdzenie czy ostatnie 8 kolumn to tylko same linki URL

In [7]:
for cecha in col4:
    wart = data[cecha].dropna().astype(str).str[0:8].unique()
    print('Pierwsze unikalne 8 znaków w columnie {}: {}'.format(cecha, wart))

Pierwsze unikalne 8 znaków w columnie Ulotka: ['https://']
Pierwsze unikalne 8 znaków w columnie Charakterystyka: ['https://']
Pierwsze unikalne 8 znaków w columnie Etykieto-ulotka: ['https://']
Pierwsze unikalne 8 znaków w columnie Ulotka importu równoległego: ['https://']
Pierwsze unikalne 8 znaków w columnie Etykieto-ulotka importu równoległego: ['https://']
Pierwsze unikalne 8 znaków w columnie Oznakowanie opakowań importu równoległego: ['https://']
Pierwsze unikalne 8 znaków w columnie Materiały edukacyjne dla osoby wykonującej zawód medyczny: ['https://']
Pierwsze unikalne 8 znaków w columnie Materiały edukacyjne dla pacjenta: ['https://']


In [8]:
data = data.drop(columns=["Ulotka", "Charakterystyka", "Etykieto-ulotka", "Ulotka importu równoległego",
                          "Etykieto-ulotka importu równoległego", "Oznakowanie opakowań importu równoległego",
                          "Materiały edukacyjne dla osoby wykonującej zawód medyczny", "Materiały edukacyjne dla pacjenta"])

### Wartości brakujace

Istnieją 2 rodzaje wartości brakujących; '-' i NaN. Należy więc zamienić - na NaN przed wykonaniem następnego kroku

In [9]:
data = data.replace('-', np.nan)

Skoro nie istnieje lub nie jest odnotowana nazwa powszechnie stosowana oraz fakt, że zajmują mały odsetek danych psoatnowiałam
je usunąć

In [10]:
data = data.dropna(subset=['Nazwa powszechnie stosowana'])

In [11]:
data['Zakaz stosowania u zwierząt'].unique()

array([nan, 'Tak'], dtype=object)

Dla tej kolumny zastapimy wszytskie wartości NaN wartością 'Nie'

In [12]:
data['Zakaz stosowania u zwierząt'] = data['Zakaz stosowania u zwierząt'].fillna("Nie")

In [13]:
data['Nazwa poprzednia produktu'] = data['Nazwa poprzednia produktu'].fillna("None")

In [14]:
def uzupelnij_dgto(row):
    if pd.isna(row['Droga podania - Gatunek - Tkanka - Okres karencji']):
        return najczestsze_dgto.get(row['Postać farmaceutyczna'], np.nan)
    return row['Droga podania - Gatunek - Tkanka - Okres karencji']

In [15]:
data['Droga podania - Gatunek - Tkanka - Okres karencji'] = data[
'Droga podania - Gatunek - Tkanka - Okres karencji'].fillna("wziewna")

Tutaj lepiej zastąpić brakujące wartości wartośicą `Unknown`, nie widzę jak inaczej można by było to uzupełnić.

In [16]:
data['Numer pozwolenia'] = data['Numer pozwolenia'].fillna("Unknown")
data['Ważność pozwolenia'] = data['Ważność pozwolenia'].fillna("Unknown")

Braki dla `Moc` raczej wyrzucić nie wiedze sposbu w jaki można by je uzupełnić. Poza tym stanowią tylko około 3% danych.

In [17]:
data = data.dropna(subset=['Moc'])

In [18]:
data = data.dropna(subset=['Kod ATC', 'Podmiot odpowiedzialny', 'Opakowanie'])

`Substancja Czynna` jest z tego co widać połączeniem `Nazwa powszechnie stosowana` i `Moc`.

In [19]:
def uzupelnij_sc(row):
    if pd.isna(row['Substancja czynna']):
        return f"{row['Nazwa powszechnie stosowana']} {row['Moc']}"
    return row['Substancja czynna']

In [20]:
data['Substancja czynna'] = data.apply(uzupelnij_sc, axis=1)

In [21]:
data['Nazwa wytwórcy'] = data['Nazwa wytwórcy'].fillna("Unknown")
data['Kraj wytwórcy'] = data['Kraj wytwórcy'].fillna("Unknown")
data['Nazwa importera'] = data['Nazwa importera'].fillna("Unknown")
data['Kraj importera'] = data['Kraj importera'].fillna("Unknown")
data['Nazwa wytwórcy/importera'] = data['Nazwa wytwórcy/importera'].fillna("Unknown")
data['Kraj wytwórcy/importera'] = data['Kraj wytwórcy/importera'].fillna("Unknown")
data['Podmiot odpowiedzialny w kraju eksportu'] = data['Podmiot odpowiedzialny w kraju eksportu'].fillna("Unknown")
data['Kraj eksportu'] = data['Kraj eksportu'].fillna("Unknown")
data['Podstawa prawna wniosku'] = data['Podstawa prawna wniosku'].fillna("Unknown")